# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² protein mass spectrometry dataset using the `mlcroissant` library, following the MLCommons Croissant data standard.

### Dataset Source
This dataset is sourced via a Croissant schema URL, ensuring reproducible and FAIR (Findable, Accessible, Interoperable, Reusable) data processing.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Metadata includes information about records sets, fields, and their `@id`s for referencing.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print out dataset metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Dataset ID (@id): {dataset.metadata.id}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All entities are referenced by their Croissant `@id`. 

Let's enumerate available record sets (tables) and their schemas using the Croissant metadata.

In [ ]:
# Get all record sets
record_sets = list(dataset.metadata.record_sets)
print(f"Record Sets Found: {len(record_sets)}\n")
for rset in record_sets:
    print(f"Record Set Name: {rset.name}\n  @id: {rset.id}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use the record set and field `@id`s for referencing.

> Note: The following step will load *all* data from each record set found. For large datasets, you may wish to process them one at a time or limit the number of records.

In [ ]:
# Collect record_set @id's
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]
dataframes = {}
for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        if records:
            dataframes[rset_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[rset_id])} records for record set '{rset_id}'")
        else:
            print(f"No records found for record set '{rset_id}'")
    except Exception as e:
        print(f"Error loading record set '{rset_id}': {e}")

# Show columns of first non-empty data frame
for rset_id, df in dataframes.items():
    print(f"\nColumns for record set '{rset_id}':")
    print(df.columns.tolist())
    print(df.head())
    break  # Show only the first loaded record set as an example

## 4. Exploratory Data Analysis (EDA)
Explore and process the data. Pick a numeric field for transformations, such as filtering, normalization, and grouping - always using field `@id` as reference. 

Change the `numeric_field_id` and `group_field_id` below to match IDs from your data overview.

In [ ]:
# Select appropriate record set and fields by @id (use the printed output above)
selected_record_set = None
# Find a record set with data
for k in dataframes:
    if not dataframes[k].empty:
        selected_record_set = k
        break

# Example: inspect all numeric columns
print("Numeric fields available in selected data:")
numeric_candidates = []
for col in dataframes[selected_record_set].columns:
    # Try to infer numeric by dtype
    if pd.api.types.is_numeric_dtype(dataframes[selected_record_set][col]):
        numeric_candidates.append(col)
        print(f"- {col}")
if not numeric_candidates:
    print("No numeric candidates found.")
else:
    # Use the first numeric field found as a default example
    numeric_field_id = numeric_candidates[0]
    print(f"\nUsing '{numeric_field_id}' as the numeric field example.")

    # Filter out rows where the numeric field > threshold
    threshold = dataframes[selected_record_set][numeric_field_id].mean()
    filtered_df = dataframes[selected_record_set][dataframes[selected_record_set][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    group_field_id = None
    # Find a non-numeric field
    for col in dataframes[selected_record_set].columns:
        if not pd.api.types.is_numeric_dtype(dataframes[selected_record_set][col]):
            group_field_id = col
            break

    if group_field_id:
        print(f"\nGrouping by field '{group_field_id}':")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib. Example: histogram and boxplot of the chosen numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_candidates:
    field = numeric_field_id
    df = dataframes[selected_record_set]

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[field].dropna(), kde=True)
    plt.title(f"Histogram of {field}")

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[field].dropna())
    plt.title(f"Boxplot of {field}")

    plt.tight_layout()
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=field, data=df)
        plt.title(f"{field} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the FAIR-compliant Croissant schema, with all entities referenced by their `@id`, to load, inspect, analyze, and visualize a molecular dataset using the `mlcroissant` library. 

- We explored record sets and fields, referenced always by their Croissant `@id`.
- Data was loaded dynamically using these identifiers, ensuring portability.
- Basic EDA and visualizations were performed on example numeric fields.

For in-depth scientific analysis, continue with statistical tests or predictive modeling, using the record set and field `@id`s as references throughout future workflows.